# MTL-coupled Purcell-filter geometry

Selected geometry cells from the supplied Project15 iteration. This is a later geometry variant, not an exact regeneration of Project14. Requires a Qiskit Metal desktop environment. HFSS is not started by this notebook.


In [ ]:
# ==============================================================================
# CELL 1: Environment and custom MTL component
# ==============================================================================
import numpy as np
import qiskit_metal as metal
from qiskit_metal import designs, Dict, MetalGUI, draw, QComponent
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee



In [ ]:
from qiskit_metal.qlibrary.lumped.cap_3_interdigital import Cap3Interdigital

In [ ]:

design = designs.DesignPlanar()
design.overwrite_enabled = True
gui = MetalGUI(design)

In [ ]:
design.chips.main.size.size_x = '4 mm'
design.chips.main.size.size_y = '2 mm'
design.chips.main.size.size_z = '-270 um'
design.chips.main.size.sample_holder_top = '500um'
design.chips.main.size.sample_holder_bottom = '800um'

design.chips.main

In [ ]:
class FourPortMTL(QComponent):
    """Custom 4-port straight parallel MTL for intrinsic Purcell filters."""
    default_options = Dict(
        coupling_length='318 um', w='5um', s='7.5um', d='5.5um',
        pos_x='0mm', pos_y='0mm', orientation='0'
    )

    def make(self):
        p = self.parse_options()
        L, w, s, d = p.coupling_length, p.w, p.s, p.d
        y_offset = (d + w + 2*s) / 2.0  

        # 1. Draw the parallel straight paths at the origin
        ro_line = draw.LineString([(-L/2, y_offset), (L/2, y_offset)])
        fil_line = draw.LineString([(-L/2, -y_offset), (L/2, -y_offset)])
        
        # 2. Apply Orientation and Position transformations
        ro_line = draw.translate(draw.rotate(ro_line, p.orientation, origin=(0,0)), p.pos_x, p.pos_y)
        fil_line = draw.translate(draw.rotate(fil_line, p.orientation, origin=(0,0)), p.pos_x, p.pos_y)
        
        # 3. Add to QGeometry
        self.add_qgeometry('path', {'ro_trace': ro_line, 'fil_trace': fil_line}, width=p.w, layer=1)
        self.add_qgeometry('path', {'ro_cut': ro_line, 'fil_cut': fil_line}, width=w + 2*s, subtract=True, layer=1)

        # 4. Helper function to transform pin coordinates
        def xform(pts):
            line = draw.LineString(pts)
            line = draw.translate(draw.rotate(line, p.orientation, origin=(0,0)), p.pos_x, p.pos_y)
            return np.array(line.coords)

        # 5. Add Pins using transformed coordinates
        self.add_pin('readout_left',  points=xform([[-L/2, y_offset - w/2], [-L/2, y_offset + w/2]]), width=p.w)
        self.add_pin('filter_left',   points=xform([[-L/2, -y_offset - w/2],[-L/2, -y_offset + w/2]]),width=p.w)
        self.add_pin('readout_right', points=xform([[L/2, y_offset + w/2],  [L/2, y_offset - w/2]]), width=p.w)
        self.add_pin('filter_right',  points=xform([[L/2, -y_offset + w/2], [L/2, -y_offset - w/2]]),width=p.w)


cpw_w = '5um'
cpw_s = '7.5um'

# ==============================================================================
# CELL 2: Core Components & Feedline Placement
# ==============================================================================
# Central MTL section (318 um active default, d = 5.5 um)
tee_mtl = FourPortMTL(design, 'tee_mtl', options=dict(pos_x='0 mm', pos_y='0 mm'))

# Transmon Q1 - Positioned securely to the left, facing right (orientation=0)
# q1 = TransmonPocket(design, 'Q1', options=dict(
#     pos_x='0.15 mm', pos_y='0.8 mm', orientation='-90',
#     connection_pads=dict(readout=dict(pad_width='125 um', pad_gap='15 um', cpw_width=cpw_w, cpw_gap=cpw_s))
# ))

# Feedline & Launchpads
port1 = LaunchpadWirebondDriven(design, 'port1', options=dict(pos_x='-1.5 mm', pos_y='-0.5 mm', orientation='0', trace_width=cpw_w, trace_gap=cpw_s))
port2 = LaunchpadWirebondDriven(design, 'port2', options=dict(pos_x='1.5 mm', pos_y='-0.5 mm', orientation='180', trace_width=cpw_w, trace_gap=cpw_s))
port3 = LaunchpadWirebondDriven(design, 'port3', options=dict(pos_x='0.6 mm', pos_y='0.5 mm', orientation='180', trace_width=cpw_w, trace_gap=cpw_s))

port3_cap = Cap3Interdigital(design, 'port3_cap', options=dict(pos_x='0.48 mm', pos_y='0.5 mm', orientation='90', finger_length='50um', finger_width='5um', finger_gap='5um', num_fingers=3))

# Coupled Line Tee for Feedline
tee_feedline = CoupledLineTee(design, 'tee_feedline', options=dict(
    pos_x='-0.3 mm', pos_y='-0.5 mm', orientation='180', mirror=False,
    coupling_length='100 um', down_length='50 um', 
    prime_width=cpw_w, prime_gap=cpw_s, second_width=cpw_w, second_gap=cpw_s, coupling_space='5um',
    hfss_wire_bonds = True
))

feed1 = RouteStraight(design, 'feed1', options=dict(pin_inputs=dict(start_pin=dict(component='port1', pin='tie'),
                                                                    end_pin=dict(component='tee_feedline', pin='prime_end')),
                                                                    hfss_wire_bonds = True,
                                                                    trace_width=cpw_w, trace_gap=cpw_s))
feed2 = RouteStraight(design, 'feed2', options=dict(pin_inputs=dict(start_pin=dict(component='tee_feedline', pin='prime_start'),
                                                                    end_pin=dict(component='port2', pin='tie')),
                                                                    hfss_wire_bonds = True,
                                                                    trace_width=cpw_w, trace_gap=cpw_s))

feed3 = RouteStraight(design, 'feed3', options=dict(pin_inputs=dict(start_pin=dict(component='port3_cap', pin='b'),
                                                                    end_pin=dict(component='port3', pin='tie')),
                                                                    hfss_wire_bonds = True,
                                                                    trace_width=cpw_w, trace_gap=cpw_s))

# # Shorting Grounds
short_ro = ShortToGround(design, 'short_ro', options=dict(pos_x='-0.3 mm', pos_y='0.27 mm', orientation='180'))\


# # Placed perfectly for an exact straight line to the filter short leg (lp^s = 318um)
# # x = L/2 (0.8295) + 0.318 = 1.1475 mm
short_fil = ShortToGround(design, 'short_fil', options=dict(pos_x='0.2 mm', pos_y='-0.3 mm', orientation='0'))


# ==============================================================================
# CELL 3: Resonator legs for this geometry variant
# ==============================================================================
# # Standardized meander safety options
# m_opts = dict(spacing='40um', asymmetry='0um')
l_opts = dict(start_straight='20um', end_straight='20um')
f_rad = '15um'

# 1. Filter short leg (l_p^s = 1659 um) -> Straight line avoids meander crashing on short distances
filter_ls = RouteMeander(design, 'filter_ls', options=dict(
    pin_inputs=dict(start_pin=dict(component='tee_mtl', pin='filter_right'), end_pin=dict(component='short_fil', pin='short')),
    total_length='1.759 mm', fillet = '15um',
    trace_width=cpw_w, trace_gap=cpw_s,
    hfss_wire_bonds = True,
    meander = dict(spacing='50um', asymmetry='-100um'), lead = dict(start_straight='20um', end_straight='20um')
))

# 2. Filter open leg (l_p^o = 759 um) - feedline coupler length 150um
filter_lo = RouteMeander(design, 'filter_lo', options=dict(
    pin_inputs=dict(start_pin=dict(component='tee_feedline', pin='second_end'), end_pin=dict(component='tee_mtl', pin='filter_left')),
    total_length='0.609 mm', trace_width=cpw_w, trace_gap=cpw_s, fillet=f_rad,
    hfss_wire_bonds = True, 
    meander=dict(spacing='390um', asymmetry='-60um'), lead=dict(start_straight='20um', end_straight='50um')
))

# # # 3. Readout open leg (l_r^o = 974 um) - qubit coupling length 320um
readout_lo = RouteMeander(design, 'readout_lo', options=dict(
    pin_inputs=dict(start_pin=dict(component='tee_mtl', pin='readout_right'), end_pin=dict(component='port3_cap', pin='a')),
    total_length='0.874 mm', trace_width=cpw_w, trace_gap=cpw_s, fillet=f_rad,
    hfss_wire_bonds = True,
    meander=dict(spacing='200um', asymmetry='-100um'), lead=l_opts
))

# # 4. Readout short leg (l_r^s = 1617 um)
readout_ls = RouteMeander(design, 'readout_ls', options=dict(
    pin_inputs=dict(start_pin=dict(component='tee_mtl', pin='readout_left'), end_pin=dict(component='short_ro', pin='short')),
    total_length='1.717 mm', trace_width=cpw_w, trace_gap=cpw_s, fillet=f_rad,
    hfss_wire_bonds = True,
    meander=dict(spacing='50um', asymmetry='-100um'), lead=l_opts
))

gui.rebuild()
gui.autoscale()